# NB09 — Yoko LAS registration from QGIS control points

**Musanga / Final** · source: `Data/LAS/raw/Yoko.las` · controls: `Data/LAS/raw/Yoko.points` · target: `Data/DSM/Yoko_kvo_georef_DSM.tif` · output: `Data/Las/Yoko_aligned.las`.

Fits an **affine approximation** of the original QGIS georeferencing from enabled `sourceX,sourceY → mapX,mapY` control pairs. Unlike the earlier ±40 m translation optimization, this supports rotation, scale and shear. It does **not** guarantee equivalence with a QGIS polynomial or thin-plate-spline transformation.

**Safeguards:** direct paths only, streaming LAS access, memory-bounded DSM, point-control diagnostics, DSM-vs-LAS before/after checks, and export OFF by default pending visual review. Horizontal coordinates change; Z is preserved.

In [ ]:
!pip -q install laspy lazrs rasterio pyproj scipy pandas matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import copy, json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import laspy, rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
from scipy.ndimage import map_coordinates
from pyproj import CRS

ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
LAS = ROOT/'Data/LAS/raw/Yoko.las'
POINTS = ROOT/'Data/LAS/raw/Yoko.points'
DSM = ROOT/'Data/DSM/Yoko_kvo_georef_DSM.tif'
OUT = ROOT/'Data/Las/Yoko_aligned.las'
QA_DIR = ROOT/'Data/Las'

GRID_RES_M = 3.0
MAX_CELLS = 2_000_000
CHUNK_SIZE = 200_000
MAX_SAMPLE_CELLS = 30_000
RNG_SEED = 9

# Review QA and set True in a separate run ONLY if source frame and fit make sense.
EXPORT_ALIGNED_LAS = False
for f in (LAS,POINTS,DSM):
    if not f.is_file(): raise FileNotFoundError(str(f))
QA_DIR.mkdir(parents=True,exist_ok=True)
print('LAS:',LAS,'\nGCP:',POINTS,'\nDSM:',DSM,'\nOutput:',OUT)

In [ ]:
# Read QGIS .points. Its first line is a CRS comment, not a CSV record.
with open(POINTS,encoding='utf-8-sig') as stream:
    first=stream.readline().strip()
gcp=pd.read_csv(POINTS,comment='#')
required={'mapX','mapY','sourceX','sourceY','enable'}
if not required.issubset(gcp): raise ValueError(f'Missing GCP columns: {required-set(gcp)}')
gcp=gcp[gcp.enable.astype(str).str.strip().isin(['1','1.0','True','true'])].copy()
gcp=gcp.replace([np.inf,-np.inf],np.nan).dropna(subset=['mapX','mapY','sourceX','sourceY'])
if len(gcp)<6: raise ValueError('Need at least 6 valid enabled GCPs')
source=gcp[['sourceX','sourceY']].to_numpy(dtype=np.float64)
target=gcp[['mapX','mapY']].to_numpy(dtype=np.float64)
origin=source.mean(axis=0)
A=np.column_stack((source-origin,np.ones(len(source))))
coef=np.linalg.lstsq(A,target,rcond=None)[0]
predict=lambda x,y: ((np.column_stack((np.asarray(x)-origin[0],np.asarray(y)-origin[1],
                                    np.ones_like(np.asarray(x)))) @ coef))
pred=predict(source[:,0],source[:,1])
error=np.linalg.norm(pred-target,axis=1)
rmse=float(np.sqrt(np.mean(error**2)))
print('Enabled GCPs:',len(gcp),'affine horizontal RMSE:',round(rmse,3),'m')
print('Median:',round(np.median(error),3),'m; P95:',round(np.percentile(error,95),3),'m')
print('Linear transform X:',coef[:,0])
print('Linear transform Y:',coef[:,1])
if rmse>3.0:
    warnings.warn('Affine control-point residuals exceed 3 m; inspect QGIS transform type.')
# Ten deterministic random folds for out-of-sample accuracy.
rng=np.random.default_rng(RNG_SEED); order=rng.permutation(len(gcp))
folds=np.array_split(order, min(5,len(gcp)))
cv_error=np.empty(len(gcp))
for test in folds:
    train=np.setdiff1d(np.arange(len(gcp)),test)
    p=np.linalg.lstsq(A[train],target[train],rcond=None)[0]
    cv_error[test]=np.linalg.norm(A[test]@p-target[test],axis=1)
print('5-fold GCP cross-validation RMSE:',np.sqrt(np.mean(cv_error**2)).round(3),'m')
plt.figure(figsize=(6,4))
plt.hist(error,bins=30)
plt.xlabel('Horizontal affine GCP fit error (m)');plt.ylabel('Control points')
plt.title('Yoko QGIS source → map control fit');plt.show()

In [ ]:
with laspy.open(LAS) as src:
    h=src.header
    bounds_min,bounds_max=h.mins.copy(),h.maxs.copy()
    las_crs=h.parse_crs()
    n_points=h.point_count
with rasterio.open(DSM) as ds:
    dsm_crs=ds.crs
    db=ds.bounds
    dres=ds.res
print(f'LAS: {n_points:,} points, bounds',bounds_min,bounds_max,'CRS:',las_crs)
print('DSM:',dsm_crs,'bounds:',db,'resolution:',dres)
if dsm_crs is None or not dsm_crs.is_projected:
    raise ValueError('Reference DSM must have projected metric CRS')
if not np.all(np.isclose([a.unit_conversion_factor for a in CRS.from_user_input(dsm_crs).axis_info[:2]],1)):
    raise ValueError('Expected DSM coordinates in metres.')
if las_crs is not None and not CRS.from_user_input(las_crs).equals(CRS.from_user_input(dsm_crs)):
    raise ValueError('LAS and DSM CRS metadata differ. This affine fit is not CRS reprojection.')
if first.startswith('#CRS:'):
    try:
        qgis_crs=CRS.from_user_input(first[5:].strip())
        if not qgis_crs.equals(CRS.from_user_input(dsm_crs)):
            raise ValueError('QGIS GCP target CRS differs from reference DSM CRS')
    except ValueError:
        raise
# Check that LAS is plausibly in SOURCE (unwarped) coordinates.
las_center=np.array([(bounds_min[0]+bounds_max[0])/2,(bounds_min[1]+bounds_max[1])/2])
source_box=(source.min(0),source.max(0))
map_box=(target.min(0),target.max(0))
print('GCP source bounds:',*source_box)
print('GCP map bounds:',*map_box)
print('LAS centre:',las_center,'mapped centre:',predict([las_center[0]],[las_center[1]])[0])
if not (np.all(las_center>source_box[0]-3000) and np.all(las_center<source_box[1]+3000)):
    raise RuntimeError('LAS centre is far from GCP source envelope. DO NOT transform until coordinate frames are checked.')

In [ ]:
# Build a LAS upper-surface grid in the raw source coordinate frame.
xmin,ymin,xmax,ymax=bounds_min[0],bounds_min[1],bounds_max[0],bounds_max[1]
res=float(GRID_RES_M)
while np.ceil((xmax-xmin)/res)*np.ceil((ymax-ymin)/res)>MAX_CELLS:
    res*=1.5
cols=int(np.ceil((xmax-xmin)/res)); rows=int(np.ceil((ymax-ymin)/res))
print('Surface grid:',rows,cols,'resolution:',res,'m')
flat=np.full(rows*cols,-np.inf,dtype=np.float32)
with laspy.open(LAS) as reader:
    for points in reader.chunk_iterator(CHUNK_SIZE):
        x=np.asarray(points.x); y=np.asarray(points.y); z=np.asarray(points.z)
        cc=np.floor((x-xmin)/res).astype(np.int64)
        rr=np.floor((ymax-y)/res).astype(np.int64)
        good=np.isfinite(z)&(cc>=0)&(cc<cols)&(rr>=0)&(rr<rows)
        np.maximum.at(flat,rr[good]*cols+cc[good],z[good].astype(np.float32))
grid=flat.reshape(rows,cols)
rr,cc=np.where(np.isfinite(grid))
if len(rr)<300: raise RuntimeError('Too few valid LAS grid cells')
rng=np.random.default_rng(RNG_SEED)
if len(rr)>MAX_SAMPLE_CELLS:
    idx=rng.choice(len(rr),MAX_SAMPLE_CELLS,replace=False)
    rr,cc=rr[idx],cc[idx]
z=grid[rr,cc].astype(np.float64)
px=xmin+(cc+0.5)*res
py=ymax-(rr+0.5)*res
mapped=predict(px,py)
# Load only a *coarse raster* covering source and mapped samples for fair comparison.
allx=np.r_[px,mapped[:,0]]; ally=np.r_[py,mapped[:,1]]
margin=3*res
left,right=allx.min()-margin,allx.max()+margin
bottom,top=ally.min()-margin,ally.max()+margin
dresample=res
while np.ceil((right-left)/dresample)*np.ceil((top-bottom)/dresample)>MAX_CELLS:
    dresample*=1.5
ncol=int(np.ceil((right-left)/dresample)); nrow=int(np.ceil((top-bottom)/dresample))
affine=from_origin(left,top,dresample,dresample)
dsm=np.full((nrow,ncol),np.nan,dtype=np.float32)
with rasterio.open(DSM) as ds:
    reproject(rasterio.band(ds,1),dsm,src_transform=ds.transform,
              src_crs=ds.crs,src_nodata=ds.nodata,
              dst_transform=affine,dst_crs=ds.crs,dst_nodata=np.nan,
              resampling=Resampling.average,warp_mem_limit=64,num_threads=2)
print('DSM coarse array:',dsm.shape, f'{dsm.nbytes/2**20:.1f} MiB')
def comparison(x,y):
    col=(x-affine.c)/affine.a-.5
    row=(y-affine.f)/affine.e-.5
    d=map_coordinates(dsm,[row,col],order=1,mode='constant',cval=np.nan,prefilter=False)
    v=np.isfinite(d)&np.isfinite(z)
    e=d[v]-z[v]
    return {'n':int(v.sum()),'median_bias_m':float(np.median(e)) if len(e) else None,
            'centered_MAD_m':float(np.median(np.abs(e-np.median(e)))) if len(e) else None},e
before,eb=comparison(px,py)
after,ea=comparison(mapped[:,0],mapped[:,1])
print('Unaligned LAS vs reference DSM:',before)
print('GCP-affine LAS vs reference DSM:',after)
if min(before['n'],after['n'])<300:
    warnings.warn('Insufficient overlapping cells to validate surfaces.')
improvement=(before['centered_MAD_m']-after['centered_MAD_m'])/before['centered_MAD_m'] if (before['centered_MAD_m'] and after['centered_MAD_m'] is not None) else float('nan')
print(f'Centered-MAD change: {100*improvement:+.1f}% (positive is better)')
if not np.isfinite(improvement) or improvement<0.05:
    warnings.warn('DSM comparison does NOT demonstrate clear improvement. Do not export without independent review.')

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].scatter(source[:,0],source[:,1],c=error,s=14)
ax[0].set(title='QGIS GCP affine residual (m)',xlabel='sourceX',ylabel='sourceY',aspect='equal')
for a,e,name in ((ax[1],eb,'Before'),(ax[2],ea,'After affine')):
    if len(e):
        lo,hi=np.percentile(e,[2,98])
        a.hist(e,bins=75,range=(lo,hi))
        a.axvline(np.median(e),ls='--',color='red')
    a.set(title=name,xlabel='DSM − LAS upper-surface Z (m)')
plt.tight_layout()
qa=QA_DIR/'09_Yoko_GCP_affine_QA.png'
plt.savefig(qa,dpi=180,bbox_inches='tight')
plt.show()
summary={'method':'Affine GCP source→map approximation','gcp_count':len(gcp),
         'gcp_rmse_m':rmse,'gcp_cv_rmse_m':float(np.sqrt(np.mean(cv_error**2))),
         'coef_centered':coef.tolist(),'origin_source_xy':origin.tolist(),
         'before':before,'after':after,'mad_improvement_fraction':float(improvement),
         'export_default':'disabled for review'}
report=QA_DIR/'09_Yoko_GCP_affine_parameters.json'
report.write_text(json.dumps(summary,indent=2))
print('QA:',qa,'\nParameters:',report)

In [ ]:
# Export only after confirming the source LAS is in the original QGIS SOURCE frame.
# After QA review, change EXPORT_ALIGNED_LAS=True in the configuration cell.
if EXPORT_ALIGNED_LAS:
    if not np.isfinite(improvement) or improvement<0.05:
        raise RuntimeError('Export blocked: GCP transform did not clearly improve DSM correspondence.')
    if rmse>3:
        raise RuntimeError('Export blocked: GCP affine residual RMSE >3 m.')
    if OUT.exists():
        raise FileExistsError(f'{OUT} already exists; never overwrite automatically.')
    temp=OUT.with_name(OUT.stem+'_writing.las')
    if temp.exists():
        raise FileExistsError(f'Partial output exists: {temp}; inspect/remove it first.')
    try:
        with laspy.open(LAS) as reader:
            header=copy.deepcopy(reader.header)
            with laspy.open(temp,mode='w',header=header) as writer:
                for chunk in reader.chunk_iterator(CHUNK_SIZE):
                    xy=predict(np.asarray(chunk.x),np.asarray(chunk.y))
                    chunk.x=xy[:,0];chunk.y=xy[:,1]
                    # Z deliberately unchanged
                    writer.write_points(chunk)
        with laspy.open(temp) as check:
            assert check.header.point_count==n_points
        temp.rename(OUT)
        print('EXPORTED:',OUT)
    except Exception:
        print('Export interrupted. Check temporary LAS:',temp)
        raise
else:
    print('QA ONLY — no new LAS written. Inspect fit, overlap and histograms before enabling export.')

## Reuse for other sites
Once Yoko is independently verified, reuse this procedure by changing **only** the exact LAS, `.points`, DSM and output paths for each site. Never reuse the Yoko control-point transform for another location.

**Limitations:** The QGIS `.points` file lists correspondences, not necessarily the georeferencer's transformation model. An affine fit can approximate those points but can differ from higher-order polynomial/TPS results away from control points. Canopy mismatches affect DSM-based validation. Always inspect spatial overlay of the corrected point cloud before calling the alignment final.